# CUSTOMHyS in practice
CUSTOMHyS builds optimisation algorithms by combining **search operators**, and a **hyper-heuristic** can pick that combination for you. This notebook follows the order in which the framework is usually used: benchmark functions, populations, operators, metaheuristics, hyper-heuristics and, finally, a problem written from scratch (Rosenbrock).

Original tutorial by Jorge Mario Cruz-Duarte, based on the [CUSTOMHyS paper in SoftwareX](https://doi.org/10.1016/j.softx.2020.100628).
English · [Español](Tutorial_Spanish.ipynb) · [Português (Brasil)](Tutorial_Portuguese_BR.ipynb)

## 0. Setup
Requires Python 3.10–3.12. From the repository root:

```shell
python -m pip install -e ".[examples]"
```

LaTeX and TensorFlow are not needed (the TensorFlow warning on import can be ignored). All problems are **minimisation** problems.

Iterations, population sizes and replicas are small so the notebook runs quickly. For more reliable results, raise them and use more seeds.

If you change the framework's code, restart the kernel: re-importing does not reload modules that are already loaded, not even with `pip install -e`. The cell below prints which `operators.py` is in use.

In [ ]:
import sys
import ast
import random
from importlib.resources import files
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import customhys
from customhys import benchmark_func as bf
from customhys import hyperheuristic as hh
from customhys import metaheuristic as mh
from customhys import operators as op
from customhys import population as pp
from customhys import tools as tl

print("Python:", sys.executable)
print("operators.py:", op.__file__)

%matplotlib inline
plt.rcParams.update({"text.usetex": False, "figure.figsize": (7, 4)})
SEED = 42
NUM_AGENTS = 32
NUM_ITERATIONS = 60
NUM_REPLICAS = 3
HH_STEPS = 8
np.random.seed(SEED)
random.seed(SEED)
print("CUSTOMHyS", customhys.__version__)

labels = {
    "iteration": "Iteration",
    "step": "Recorded operator step",
    "fitness": "Objective value",
    "best": "Best so far",
    "current": "Current",
    "position": "Position (original domain)",
    "init": "Initialisation",
    "operator": "Operator",
    "parameters": "Parameters",
    "selector": "Selector",
    "replica": "Replica",
    "performance": "Performance (median + IQR)",
    "centroid": "Centroid (normalised)",
    "radius": "Radius (normalised)",
}

labels.update(
    {
        "snapshot": "Population",
        "record": "Operator application",
        "trajectory": "Five individuals and the best point",
        "occupancy": "Where the population remained",
        "count": "Recorded positions",
        "optimum": "Known minimum",
        "convergence": "Best objective value",
        "init_title": "Starting points",
        "unique": "Distinct points",
        "edge": "Points on a boundary (%)",
        "bins": "Occupied cells (8 × 8)",
        "start": "Initial",
        "end": "Final",
        "distance": "Distance to (1, 1)",
        "search": "Algorithm search",
        "replay": "Chosen algorithm",
        "initial": "Initial objective",
    }
)


## 1. Benchmark functions
The `benchmark_func` module collects the test functions. Each one is created with its number of dimensions and comes with a known domain and minimum. Rastrigin has many local minima and its global minimum is zero at the origin; `plot()` draws the two-dimensional surface.

In [ ]:
print(len(bf.__all__), bf.__all__)
fun = bf.Rastrigin(2)
fun.plot(samples=40, resolution=100)


`get_function_value` evaluates a single point. Here it builds the `X, Y, Z` mesh to draw the surface by hand. We will reuse that mesh as the background for the population plots.

In [ ]:
bounds = np.asarray(fun.get_search_range()).T
X, Y = np.meshgrid(*(np.linspace(low, high, 40) for low, high in bounds))
Z = np.array(
    [fun.get_function_value([x, y]) for x, y in zip(X.ravel(), Y.ravel(), strict=True)]
).reshape(X.shape)
fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.plot_surface(X, Y, Z, cmap="viridis", alpha=0.8)
ax.set(xlabel="$x_1$", ylabel="$x_2$", zlabel=labels["fitness"])
plt.show()


Domain, scale, offset and noise change through the `set_*` methods. We use a separate Ackley instance for this. With noise, evaluating the same point twice gives different values, so the other examples use unmodified functions.

In [ ]:
modified_fun = bf.Ackley1(2)
modified_fun.set_search_range([-1.0, -1.0], [1.0, 1.0])
modified_fun.set_offset_function(-2)
modified_fun.set_scale_function(2)
modified_fun.set_noise_level(0.1)
modified_fun.set_offset_domain([0.5, 0.5])
modified_fun.plot(samples=40)


## 2. Population and initialisation
`Population` stores positions, each individual's objective value and the best records (per individual and global). Internally positions live in `[-1, 1]`; `get_positions()` converts them to the problem's domain.

The usual order is: initialise (`initialise_positions`), evaluate (`evaluate_fitness`) and update the population, personal-best and global-best records (`update_positions`). Operators expect those records to be filled in. The table shows the same coordinate on both scales.

In [ ]:
fun = bf.Rastrigin(2)
pop = pp.Population(fun.get_search_range(), num_agents=NUM_AGENTS)
pop.initialise_positions("lhs")
pop.evaluate_fitness(fun.get_function_value)
pop.update_positions("population", "all")
pop.update_positions("particular", "all")
pop.update_positions("global", "greedy")
display(
    pd.DataFrame(
        {
            "x1 [-1, 1]": pop.positions[:, 0],
            "x1 [-5.12, 5.12]": pop.get_positions()[:, 0],
            labels["fitness"]: pop.fitness,
        }
    ).head()
)
print(pop.get_state())


In [ ]:
def show_positions(population):
    positions = population.get_positions()
    fig, ax = plt.subplots()
    ax.contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.5)
    ax.scatter(positions[:, 0], positions[:, 1], c="tab:red", s=20)
    ax.set(xlabel="$x_1$", ylabel="$x_2$")
    plt.show()


show_positions(pop)


### 2.1 Initialisation methods
`initialise_positions` accepts several schemes. The plots below show only the starting points (no search yet): 64 individuals in a square domain. Distribution parameters refer to the unit interval, before conversion to the domain.

| Method | How it places points |
| --- | --- |
| `random` | Uniform draws. Can leave gaps and form clusters. |
| `vertex` | Vertices of nested squares; not a full Cartesian grid. |
| `lhs` | Latin hypercube: one point in each of 64 equal intervals along each axis. |
| `sobol`, `halton` | Low-discrepancy sequences that spread points regularly. `scramble=False` makes the sample reproducible. |
| `beta` | Parameters below 1 push points to the edges; above 1, to the centre. |
| `normal` | Concentrates points near the centre (mean 0.5, standard deviation 0.25 on the unit interval). |
| `lognormal`, `exponential`, `rayleigh` | Asymmetric distributions. Values are clipped to `[0, 1]` before conversion, so points pile up on the boundary; this shows in the lognormal with these parameters. |

In [ ]:
INITIAL_AGENTS = 64
initialisation_methods = [
    ("random", "random", {}),
    ("vertex", "vertex", {}),
    ("lhs", "lhs", {}),
    ("sobol", "sobol", {"scramble": False}),
    ("halton", "halton", {"scramble": False}),
    ("beta (0.5, 0.5)", "beta", {"a": 0.5, "b": 0.5}),
    ("beta (2, 2)", "beta", {"a": 2.0, "b": 2.0}),
    ("normal (0.5, 0.25)", "normal", {"mean": 0.5, "std": 0.25}),
    ("lognormal (0, 0.5)", "lognormal", {"mean": 0.0, "sigma": 0.5}),
    ("exponential (0.5)", "exponential", {"scale": 0.5}),
    ("rayleigh (0.4)", "rayleigh", {"scale": 0.4}),
]
initial_samples = {}
for name, scheme, options in initialisation_methods:
    np.random.seed(SEED)
    sample_pop = pp.Population(([-5.0, -5.0], [5.0, 5.0]), num_agents=INITIAL_AGENTS)
    sample_pop.initialise_positions(scheme, **options)
    initial_samples[name] = sample_pop.get_positions().copy()

initialisation_figures = []
for group in (initialisation_methods[:5], initialisation_methods[5:]):
    fig, axes = plt.subplots(2, 3, figsize=(11, 7), sharex=True, sharey=True, layout="constrained")
    for ax, (name, _, _) in zip(axes.flat, group):
        points = initial_samples[name]
        ax.scatter(points[:, 0], points[:, 1], s=22, alpha=0.7, color="#2563a6", edgecolors="none")
        ax.set(title=name, xlim=(-5.3, 5.3), ylim=(-5.3, 5.3), xlabel="$x_1$", ylabel="$x_2$")
        ax.set_aspect("equal")
        ax.set_xticks([-5, 0, 5])
        ax.set_yticks([-5, 0, 5])
        ax.grid(alpha=0.15)
    for ax in list(axes.flat)[len(group) :]:
        ax.set_visible(False)
    fig.suptitle(f"{labels['init_title']} · {INITIAL_AGENTS}")
    initialisation_figures.append(fig)
    plt.show()

initial_summary = []
for name, points in initial_samples.items():
    counts, _, _ = np.histogram2d(points[:, 0], points[:, 1], bins=8, range=[[-5, 5], [-5, 5]])
    initial_summary.append(
        {
            labels["init"]: name,
            labels["unique"]: len(np.unique(points, axis=0)),
            labels["edge"]: 100 * np.any(np.isclose(np.abs(points), 5.0), axis=1).mean(),
            labels["bins"]: np.count_nonzero(counts),
        }
    )
display(pd.DataFrame(initial_summary).round(1))
np.random.seed(SEED)


Overlapping points look like one in the plot, so the table counts distinct coordinates and points on the boundary. Occupied cells apply to this 8 × 8 partition and do not rank the methods. A concentrated sample makes sense when you know where the optimum is; without that, a spread-out sample covers more of the domain.

## 3. Operators and collections
An operator is a tuple `(name, parameters, selector)`. The operator proposes new positions and the selector decides which ones stay: with `greedy`, an individual only moves if its objective value improves. `initialize_*` operators restart the population, and the hyper-heuristic treats them as initialisers.

A collection is a text file with one operator per line. The default collection is read from the installed package, so it works from any directory.

In [ ]:
print(labels["operator"], op.__all__)
print(labels["selector"], op.__selectors__)
collection_text = files("customhys").joinpath("collections/default.txt").read_text(encoding="utf-8")
default_operators = [
    ast.literal_eval(line) for line in collection_text.splitlines() if line.strip()
]
print(len(default_operators))
display(
    pd.DataFrame(
        default_operators, columns=[labels["operator"], labels["parameters"], labels["selector"]]
    ).head(10)
)


`obtain_operators` lists operator families with parameter values, and `build_operators` writes a collection to `collections/` (relative to the current directory). Here we generate a two-family collection inside a temporary directory so nothing gets overwritten.

In [ ]:
import os
from tempfile import TemporaryDirectory

operator_families = [
    family
    for family in op.obtain_operators(num_vals=2)
    if family[0] in {"random_search", "spiral_dynamic"}
]
original_dir = Path.cwd()
with TemporaryDirectory() as temporary_dir:
    try:
        os.chdir(temporary_dir)
        op.build_operators(heuristics=operator_families, file_name="tutorial")
        generated_text = Path("collections/tutorial.txt").read_text(encoding="utf-8")
    finally:
        os.chdir(original_dir)
generated_operators = [
    ast.literal_eval(line) for line in generated_text.splitlines() if line.strip()
]
print(len(generated_operators))
display(
    pd.DataFrame(
        generated_operators, columns=[labels["operator"], labels["parameters"], labels["selector"]]
    ).head()
)


One operator at work: each iteration, `random_search` displaces the individuals and the `greedy` selector keeps only the improvements. Grey lines are the population's values and the highlighted line is the best result so far. This loop is, in essence, what the `Metaheuristic` class automates in the next section.

In [ ]:
operator_name, operator_params, selector = (
    "random_search",
    {"scale": 0.2, "distribution": "uniform"},
    "greedy",
)
history = [pop.fitness.copy()]
best_history = [pop.global_best_fitness]
for iteration in range(1, NUM_ITERATIONS + 1):
    pop.iteration = iteration
    getattr(op, operator_name)(pop, **operator_params)
    pop.evaluate_fitness(fun.get_function_value)
    pop.update_positions("population", selector)
    pop.update_positions("global", "greedy")
    history.append(pop.fitness.copy())
    best_history.append(pop.global_best_fitness)
history = np.asarray(history)
fig, ax = plt.subplots()
ax.plot(history, color="grey", alpha=0.15)
ax.plot(best_history, lw=2, label=labels["best"])
ax.set(xlabel=labels["iteration"], ylabel=labels["fitness"])
ax.legend()
plt.show()
show_positions(pop)


## 4. Metaheuristics
`Metaheuristic` takes a problem (`get_formatted_problem()`), a sequence of operators and the search budget, and applies the whole sequence every iteration. The example combines differential mutation and crossover on three-dimensional Rastrigin, with `lhs` initialisation.

`get_solution()` returns internal coordinates; `rescale_back` converts them to the problem's domain so you can check the result directly against the function. `historical` records the state after each operator: with two operators, every iteration produces two records, plus the initial one.

In [ ]:
problem_fun = bf.Rastrigin(3)
problem = problem_fun.get_formatted_problem()
differential_sequence = [
    (
        "differential_mutation",
        {"expression": "current-to-best", "num_rands": 2, "factor": 0.8},
        "all",
    ),
    ("differential_crossover", {"crossover_rate": 0.2, "version": "binomial"}, "greedy"),
]
met = mh.Metaheuristic(
    problem,
    differential_sequence,
    num_agents=NUM_AGENTS,
    num_iterations=NUM_ITERATIONS,
    initial_scheme="lhs",
)
met.run()
normalised_best, best_fitness = met.get_solution()
best_position = met.pop.rescale_back(normalised_best)
print("x_best =", best_position, "f_best =", best_fitness)
np.testing.assert_allclose(problem_fun.get_function_value(best_position), best_fitness, atol=1e-10)


In [ ]:
def plot_run(metaheuristic):
    recorded = metaheuristic.historical
    positions = metaheuristic.pop.rescale_back(np.asarray(recorded["position"]))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(recorded["fitness"])
    axes[0].set(xlabel=labels["step"], ylabel=labels["fitness"])
    axes[1].plot(positions)
    axes[1].set(xlabel=labels["step"], ylabel=labels["position"])
    axes[1].legend([f"$x_{i + 1}$" for i in range(metaheuristic.num_dimensions)])
    fig.tight_layout()
    plt.show()


plot_run(met)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(met.historical["centroid"])
axes[0].set(xlabel=labels["step"], ylabel=labels["centroid"])
axes[1].plot(met.historical["radius"])
axes[1].set(xlabel=labels["step"], ylabel=labels["radius"])
fig.tight_layout()
plt.show()


### 4.1 Where the population went
`historical` keeps only the best value and best position. To see the whole population, the subclass below overrides `reset_historicals` and `update_historicals`, which `run()` calls, and stores all positions after each operator and selection.

The maps show the population kept after selection, so rejected moves do not appear. Lines join consecutive records and do not mean the points along them were evaluated. The occupancy map counts repeated visits, so it also reflects how long individuals stayed in each region.

In [ ]:
from matplotlib.colors import LogNorm


class TracedMetaheuristic(mh.Metaheuristic):
    def reset_historicals(self):
        super().reset_historicals()
        self.population_history = []
        self.population_fitness = []
        self.trace_iterations = []

    def update_historicals(self):
        super().update_historicals()
        self.population_history.append(self.pop.get_positions().copy())
        self.population_fitness.append(self.pop.fitness.copy())
        self.trace_iterations.append(self.pop.iteration)


def plot_exploration(run, objective, boundaries, optimum):
    path = np.asarray(run.population_history)
    lower, upper = np.asarray(boundaries)
    grid_x, grid_y = np.meshgrid(
        np.linspace(lower[0], upper[0], 100), np.linspace(lower[1], upper[1], 100)
    )
    grid_f = np.array(
        [objective([x, y]) for x, y in zip(grid_x.ravel(), grid_y.ravel(), strict=True)]
    )
    grid_f = grid_f.reshape(grid_x.shape)
    best_path = run.pop.rescale_back(np.asarray(run.historical["position"]))
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), layout="constrained")

    def landscape(ax):
        surface = ax.contourf(
            grid_x, grid_y, np.log10(1 + grid_f), levels=22, cmap="Blues", alpha=0.65
        )
        ax.scatter(*optimum, marker="*", s=130, c="#b42318", zorder=5, label=labels["optimum"])
        ax.set(xlim=(lower[0], upper[0]), ylim=(lower[1], upper[1]), xlabel="$x_1$", ylabel="$x_2$")
        return surface

    for ax, index in zip(axes[0], [0, len(path) // 2, len(path) - 1], strict=True):
        surface = landscape(ax)
        ax.scatter(
            path[index, :, 0],
            path[index, :, 1],
            s=22,
            c="#146b5a",
            edgecolors="white",
            linewidths=0.3,
        )
        ax.scatter(*best_path[index], marker="x", s=70, c="#e07800", zorder=6, label=labels["best"])
        ax.set_title(
            f"{labels['iteration']} {run.trace_iterations[index]} · {labels['record']} {index}"
        )
    axes[0, 0].legend(fontsize=8, loc="upper left")
    fig.colorbar(surface, ax=list(axes[0]), label="log₁₀(1 + f)", shrink=0.8)

    landscape(axes[1, 0])
    for agent in range(min(5, run.num_agents)):
        axes[1, 0].plot(path[:, agent, 0], path[:, agent, 1], lw=0.7, alpha=0.45)
    axes[1, 0].plot(best_path[:, 0], best_path[:, 1], color="#e07800", lw=1.8)
    axes[1, 0].set_title(labels["trajectory"])

    visited = path.reshape(-1, 2)
    density = axes[1, 1].hist2d(
        visited[:, 0],
        visited[:, 1],
        bins=25,
        range=list(zip(lower, upper, strict=True)),
        cmap="YlGnBu",
        norm=LogNorm(),
        cmin=1,
    )
    axes[1, 1].scatter(*optimum, marker="*", c="#b42318", s=100)
    axes[1, 1].set(title=labels["occupancy"], xlabel="$x_1$", ylabel="$x_2$")
    fig.colorbar(density[3], ax=axes[1, 1], label=labels["count"], shrink=0.8)

    axes[1, 2].plot(run.historical["fitness"], color="#146b5a", lw=2)
    axes[1, 2].set(
        title=labels["convergence"], xlabel=labels["record"], ylabel="f(x)", yscale="symlog"
    )
    axes[1, 2].set_yscale('symlog', linthresh=1e-6)
    axes[1, 2].set_ylim(bottom=0)
    axes[1, 2].grid(alpha=0.15)
    plt.show()
    return fig


In [ ]:
np.random.seed(SEED)
random.seed(SEED)
visual_fun = bf.Rastrigin(2)
visual_problem = visual_fun.get_formatted_problem()
trace_rastrigin = TracedMetaheuristic(
    visual_problem,
    [("random_search", {"scale": 0.08, "distribution": "gaussian"}, "greedy")],
    num_agents=NUM_AGENTS,
    num_iterations=NUM_ITERATIONS,
    initial_scheme="lhs",
)
trace_rastrigin.run()
rastrigin_figure = plot_exploration(
    trace_rastrigin,
    visual_fun.get_function_value,
    visual_problem["boundaries"],
    optimum=(0, 0),
)


### 4.2 Variation between runs
Stochastic algorithms give different results each run, so a single run says little. Here we combine random search, spiral dynamic and gravitational search on six-dimensional Griewank, each run with a different seed. The band between P25 and P75 holds the middle half of the runs. We use only three replicas to keep the example fast.

In [ ]:
griewank = bf.Griewank(6)
griewank_problem = griewank.get_formatted_problem()
hybrid_sequence = [
    ("random_search", {"scale": 1.0, "distribution": "uniform"}, "greedy"),
    ("spiral_dynamic", {"radius": 0.9, "angle": 33.5, "sigma": 0.1}, "all"),
    ("gravitational_search", {"gravity": 0.5, "alpha": 0.2}, "metropolis"),
]
runs = []
for replica in range(NUM_REPLICAS):
    np.random.seed(SEED + replica)
    random.seed(SEED + replica)
    hybrid_met = mh.Metaheuristic(
        griewank_problem,
        hybrid_sequence,
        num_iterations=NUM_ITERATIONS,
        num_agents=NUM_AGENTS,
    )
    hybrid_met.run()
    runs.append(hybrid_met.historical["fitness"])
fitness_array = np.asarray(runs).T
final_fitness = fitness_array[-1]
display(
    pd.DataFrame(
        {labels["replica"]: np.arange(1, NUM_REPLICAS + 1), labels["fitness"]: final_fitness}
    )
)
plot_run(hybrid_met)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [3, 1]})
axes[0].plot(fitness_array, color="grey", alpha=0.4)
axes[0].plot(np.median(fitness_array, axis=1), label="P50", lw=2)
axes[0].fill_between(
    np.arange(len(fitness_array)),
    np.quantile(fitness_array, 0.25, axis=1),
    np.quantile(fitness_array, 0.75, axis=1),
    alpha=0.2,
    label="P25–P75",
)
axes[0].set(xlabel=labels["step"], ylabel=labels["fitness"])
axes[0].legend()
axes[1].boxplot(final_fitness)
axes[1].set(xlabel=labels["replica"], ylabel=labels["fitness"])
fig.tight_layout()
plt.show()


## 5. Hyper-heuristics
So far we picked the algorithm by hand. `Hyperheuristic` takes a heuristic space (a list of operators) and searches for a sequence that works well on the problem. Here the problem is two-dimensional Sphere, with minimum zero at `(0, 0)`.

Each candidate sequence runs `num_replicas` times and is scored by `median + (P75 − P25)` of the final values. The score combines the typical result and its spread; lower is better.

The collection has two initialisers. The first operator in the sequence defines how the population starts; an `initialize_*` that appears later restarts the population in the middle of the search. That behaviour must be preserved when running the sequence that was found.

In [ ]:
heuristic_space = [
    ("initialize_random", {}, "all"),
    ("initialize_lhs", {}, "all"),
    ("random_search", {"scale": 0.2, "distribution": "uniform"}, "greedy"),
    (
        "local_random_walk",
        {"probability": 0.75, "scale": 0.2, "distribution": "gaussian"},
        "greedy",
    ),
]
parameters = {
    "cardinality": 3,
    "cardinality_min": 2,
    "num_iterations": 30,
    "num_agents": NUM_AGENTS,
    "as_mh": True,
    "num_replicas": NUM_REPLICAS,
    "num_steps": HH_STEPS,
    "stagnation_percentage": 0.5,
    "max_temperature": 1.0,
    "min_temperature": 1e-6,
    "cooling_rate": 1e-3,
    "temperature_scheme": "fast",
    "acceptance_scheme": "exponential",
    "allow_weight_matrix": False,
    "trial_overflow": False,
    "learnt_dataset": None,
    "repeat_operators": True,
    "verbose": False,
    "learning_portion": 0.37,
    "solver": "static",
    "initial_scheme": "random",
}
np.random.seed(SEED)
random.seed(SEED)
hh_fun = bf.Sphere(2)
hh_problem = hh_fun.get_formatted_problem()
run_label = "tutorial-" + uuid4().hex
hyp = hh.Hyperheuristic(
    heuristic_space=heuristic_space,
    problem=hh_problem,
    parameters=parameters.copy(),
    file_label=run_label,
)
best_sol, best_perf, hist_current, hist_best = hyp.solve(save_steps=True)
best_sequence = hyp.get_operators(best_sol)
print(labels["performance"], best_perf)
display(
    pd.DataFrame(
        best_sequence, columns=[labels["operator"], labels["parameters"], labels["selector"]]
    )
)


In [ ]:
fig, ax = plt.subplots()
ax.plot(hist_current, label=labels["current"], alpha=0.7)
ax.plot(hist_best, label=labels["best"], lw=2)
ax.set(xlabel=labels["iteration"], ylabel=labels["performance"])
ax.legend()
plt.show()


### 5.1 Reading the results
Each run writes its files to `data_files/raw/<label>/`. The static solver records the initial solution and those that follow which match or improve on the best, so these are not all the attempts. `merge_json` joins the records into one file and `read_json` loads it.

In [ ]:
results_dir = Path("data_files") / "raw" / run_label
tl.merge_json(results_dir.as_posix())
full_data = tl.read_json(results_dir.with_suffix(".json").as_posix())
print(results_dir.resolve())
display(
    pd.DataFrame(
        [
            {
                "step": record["step"],
                labels["performance"]: record["performance"],
                labels["operator"]: record["encoded_solution"],
            }
            for record in full_data
        ]
    )
)
saved_fitness = full_data[-1]["details"]["fitness"]
fig, ax = plt.subplots()
ax.boxplot(saved_fitness)
ax.set(xlabel=labels["replica"], ylabel=labels["fitness"])
plt.show()


### 5.2 Running the chosen algorithm
`get_operators` turns the indices returned by the hyper-heuristic into the operator sequence. If the first operator is `initialize_*`, it becomes the `initial_scheme` and the rest follow in the same order. The run starts from a fresh population; it does not continue any of the replicas used during the search.

In [ ]:
initial_scheme = parameters["initial_scheme"]
discovered_operators = list(best_sequence)
if discovered_operators[0][0].startswith("initialize_"):
    initial_name, initial_params, _ = discovered_operators.pop(0)
    initial_scheme = (initial_name, initial_params)
discovered_met = TracedMetaheuristic(
    hh_problem,
    discovered_operators,
    num_agents=NUM_AGENTS,
    num_iterations=parameters["num_iterations"],
    initial_scheme=initial_scheme,
)
discovered_met.run()
x_normalised, f_best = discovered_met.get_solution()
x_best = discovered_met.pop.rescale_back(x_normalised)
print("x_best =", x_best, "f_best =", f_best)
np.testing.assert_allclose(hh_fun.get_function_value(x_best), f_best, atol=1e-10)
plot_run(discovered_met)


Maps from a new run of the chosen algorithm on Sphere. The hyper-heuristic searches the space of *operator sequences*; these maps show the chosen sequence searching the space of the *problem's coordinates*. They are two different spaces.

In [ ]:
sphere_figure = plot_exploration(
    discovered_met, hh_fun.get_function_value, hh_problem["boundaries"], optimum=(0, 0)
)


## 6. The CEC2005 collection
`CEC2005` gathers benchmark problems used to evaluate algorithms. F1 in five dimensions is a shifted Sphere, so its minimum is not zero. Usage is the same as before: `get_formatted_problem()` returns the problem dictionary, which goes to the metaheuristic or the hyper-heuristic.

In [ ]:
cec_fun = bf.CEC2005("F1", 5)
cec_problem = cec_fun.get_formatted_problem()
cec_met = mh.Metaheuristic(
    cec_problem,
    differential_sequence,
    num_iterations=NUM_ITERATIONS,
    num_agents=NUM_AGENTS,
)
cec_met.run()
x_normalised, cec_fitness = cec_met.get_solution()
cec_position = cec_met.pop.rescale_back(x_normalised)
print("x_best =", cec_position, "f_best =", cec_fitness)
np.testing.assert_allclose(cec_fun.get_function_value(cec_position), cec_fitness)
plot_run(cec_met)

cec_hyp = hh.Hyperheuristic(
    heuristic_space=heuristic_space,
    problem=cec_problem,
    parameters=parameters.copy(),
)
cec_sol, cec_perf, _, cec_hist_best = cec_hyp.solve(save_steps=False)
print(labels["performance"], cec_perf)
display(
    pd.DataFrame(
        cec_hyp.get_operators(cec_sol),
        columns=[labels["operator"], labels["parameters"], labels["selector"]],
    )
)


## 7. Defining your own problem: Rosenbrock
You do not need a framework class: a dictionary is enough. It holds the function (which takes the coordinate vector), the bounds `([xmin, ymin], [xmax, ymax])` and `is_constrained=True` to enforce them.

Rosenbrock has a narrow, curved valley:

$$\min_{-2 \le x \le 2,\;-1 \le y \le 3} f(x,y)=(1-x)^2+100(y-x^2)^2.$$

Both terms vanish at **(1, 1)**, where **f = 0**. Reaching the valley is usually easy; following it to the minimum is the hard part. Since the optimum is known, we can check both the function value and the distance to it.

In [ ]:
def rosenbrock(point):
    x, y = point
    return float((1 - x) ** 2 + 100 * (y - x**2) ** 2)


rosen_problem = {
    "function": rosenbrock,
    "boundaries": ([-2.0, -1.0], [2.0, 3.0]),
    "is_constrained": True,
}
assert rosenbrock([1.0, 1.0]) == 0.0
assert rosenbrock([0.0, 0.0]) == 1.0

rx, ry = np.meshgrid(np.linspace(-2, 2, 160), np.linspace(-1, 3, 160))
rz = (1 - rx) ** 2 + 100 * (ry - rx**2) ** 2
fig, ax = plt.subplots(figsize=(7, 5), layout="constrained")
surface = ax.contourf(rx, ry, np.log10(1 + rz), levels=25, cmap="Blues")
valley_x = np.linspace(-1, np.sqrt(3), 160)
ax.plot(valley_x, valley_x**2, "--", color="#e07800", lw=1.5, label="$y=x^2$")
ax.scatter(1, 1, s=150, marker="*", color="#b42318", label=labels["optimum"])
ax.set(xlabel="$x$", ylabel="$y$", title="Rosenbrock")
ax.legend()
fig.colorbar(surface, ax=ax, label="log₁₀(1 + f)")
plt.show()


### 7.1 The hyper-heuristic builds the algorithm
The search space has random moves at three scales, a local random walk and uniform sampling (`random_sample`). Initialisation is fixed to `lhs` so the search is only over operators.

The outer search tries up to 12 sequence changes (`num_steps`). Each candidate runs 80 iterations with 32 individuals and three replicas. Longer sequences use more evaluations per iteration. The performance shown summarises the replicas; it is not the function value at a single point.

In [ ]:
rosen_space = [
    ("random_search", {"scale": scale, "distribution": "gaussian"}, "greedy")
    for scale in (0.01, 0.05, 0.2)
] + [
    (
        "local_random_walk",
        {"probability": 0.75, "scale": 0.1, "distribution": "gaussian"},
        "greedy",
    ),
    ("random_sample", {}, "greedy"),
]
rosen_parameters = {
    **parameters,
    "cardinality_min": 1,
    "cardinality": 3,
    "num_iterations": 80,
    "num_steps": 12,
    "stagnation_percentage": 1.0,
    "initial_scheme": "lhs",
}
np.random.seed(SEED)
random.seed(SEED)
rosen_hyp = hh.Hyperheuristic(
    heuristic_space=rosen_space,
    problem=rosen_problem,
    parameters=rosen_parameters.copy(),
)
rosen_sol, rosen_perf, rosen_current, rosen_best = rosen_hyp.solve(save_steps=False)
rosen_sequence = rosen_hyp.get_operators(rosen_sol)
print(labels["performance"], rosen_perf)
display(
    pd.DataFrame(
        rosen_sequence, columns=[labels["operator"], labels["parameters"], labels["selector"]]
    )
)

fig, ax = plt.subplots()
ax.plot(rosen_current, label=labels["current"], alpha=0.6)
ax.plot(rosen_best, label=labels["best"], lw=2)
ax.set(title=labels["search"], xlabel=labels["iteration"], ylabel=labels["performance"])
ax.legend()
plt.show()


### 7.2 Running and checking the solution
New population, new seed. The table shows the final coordinates, the function value, the distance to `(1, 1)` and the initial value. The function is evaluated again at those coordinates to check the result. With a small budget, it may not reach the exact minimum.

In [ ]:
np.random.seed(SEED + 1)
random.seed(SEED + 1)
rosen_run = TracedMetaheuristic(
    rosen_problem,
    rosen_sequence,
    num_agents=NUM_AGENTS,
    num_iterations=rosen_parameters["num_iterations"],
    initial_scheme="lhs",
)
rosen_run.run()
rosen_normalised, rosen_fitness = rosen_run.get_solution()
rosen_position = rosen_run.pop.rescale_back(rosen_normalised)
np.testing.assert_allclose(rosenbrock(rosen_position), rosen_fitness, atol=1e-10)
display(
    pd.DataFrame(
        [
            {
                "x": rosen_position[0],
                "y": rosen_position[1],
                "f(x, y)": rosen_fitness,
                labels["distance"]: np.linalg.norm(rosen_position - np.array([1.0, 1.0])),
                labels["initial"]: rosen_run.historical["fitness"][0],
            }
        ]
    )
)
rosen_figure = plot_exploration(rosen_run, rosenbrock, rosen_problem["boundaries"], optimum=(1, 1))


### 7.3 Animation
Replays the population that was already recorded; it does not run a new optimisation. You can pause and move between frames. Frames are sampled to keep the notebook light (the last one is always included), and no video encoder is needed.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

animation_path = np.asarray(rosen_run.population_history)
animation_best = rosen_run.pop.rescale_back(np.asarray(rosen_run.historical["position"]))
frame_indices = np.unique(np.linspace(0, len(animation_path) - 1, 36, dtype=int))
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), layout="constrained")
axes[0].contourf(rx, ry, np.log10(1 + rz), levels=22, cmap="Blues", alpha=0.65)
axes[0].scatter(1, 1, marker="*", c="#b42318", s=100)
cloud = axes[0].scatter([], [], c="#146b5a", s=18)
marker = axes[0].scatter([], [], marker="x", c="#e07800", s=60)
axes[0].set(xlim=(-2, 2), ylim=(-1, 3), xlabel="$x$", ylabel="$y$")
axes[1].plot(rosen_run.historical["fitness"], color="#146b5a")
cursor = axes[1].axvline(0, color="#e07800")
axes[1].set(xlabel=labels["record"], ylabel="f(x)", title=labels["convergence"], yscale="symlog")


axes[1].set_yscale('symlog', linthresh=1e-6)
axes[1].set_ylim(bottom=0)

def update_frame(index):
    cloud.set_offsets(animation_path[index])
    marker.set_offsets(animation_best[index : index + 1])
    cursor.set_xdata([index, index])
    axes[0].set_title(f"{labels['iteration']} {rosen_run.trace_iterations[index]}")
    return cloud, marker, cursor


animation = FuncAnimation(fig, update_frame, frames=frame_indices, interval=180, repeat=False)
animation_html = animation.to_jshtml(default_mode="once")
plt.close(fig)
display(HTML(animation_html))


## Things to try
- Change only `initial_scheme`, keeping the function, operators and number of evaluations fixed, and compare the final value and the region occupied.
- In Rosenbrock, `num_steps` tests more algorithms and `num_iterations` gives each one more time.
- [quickTest](quickTest.ipynb) uses the linear-system operator.